In [11]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from statsmodels.tsa.stattools import acf
from scipy.io import arff
import pandas as pd
from hurst import compute_Hc

def Probability(Data, M, tau):
    z = np.zeros((len(Data) - (M - 1) * tau, M))

    for i in range(len(Data) - (M - 1) * tau):
        for j in range(M):
            z[i][j] = Data[i + j * tau] 

    e = np.argsort(z, axis = 1)
        
    pattern_count = {}
    for pattern in e:
        key = tuple(pattern)
        pattern_count[key] = pattern_count.get(key, 0) + 1
            
    total = sum(pattern_count.values())
    P = np.array([count / total for count in pattern_count.values()])
    return P

def Probability_U(P):
    return np.full(shape=len(P), fill_value=1/len(P))

def Probability_Const(P):
    p = np.zeros(len(P))
    p[0] = 1
    return p

def Entropy(P):
    P = P[P > 0]
    return -np.sum(P * np.log2(P))

def J(P1, P2):
    P3 = np.add(P1, P2) / 2
    return Entropy(P3) - 0.5 * (Entropy(P1) + Entropy(P2))

raw_data, meta = arff.loadarff('AbnormalHeartbeat_TEST.arff')
df = pd.DataFrame(raw_data)
feature_columns = [col for col in df.columns if col != 'target']
X = df[feature_columns].values.astype(np.float32)
Y = df['target'].values

In [12]:
Data = []
for i in range(len(X)):
    print(i)
    Data.append([])
    for m in range(8):
        Data[i].append([])
        for tau in range(12):
            data = X[i].copy()
            prob = Probability(data, m, tau)
            prob_u = Probability_U(prob)
            prob_const = Probability_Const(prob)
            if (Entropy(prob_u) > 0):
                entropy = Entropy(prob) / Entropy(prob_u)
                complexity = entropy * (J(prob, prob_u) / J(prob_u, prob_const))
                hurst, c, result = compute_Hc(X[i])
                Data[i][m].append([complexity, entropy, hurst])
            else:
                Data[i][m].append([1, 1, 1])


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204


In [21]:
maximum = 0
maximumA = 0
maximumN = 0
for m in range(4, 5):
    for tau in range(1, 5):
        print(tau)
        for i in range(len(X)):
            for j in range(len(X)):
                for o in range(len(X)):
                    WallComplexity = Data[i][m][tau][0]
                    WallEntropy = Data[j][m][tau][1]
                    WallHurst = Data[o][m][tau][2]

                    abn1 = 0
                    abn2 = 0
                    abn3 = 0
                    abn4 = 0
                    abn5 = 0
                    abn6 = 0
                    abn7 = 0
                    abn8 = 0
                    nor1 = 0
                    nor2 = 0
                    nor3 = 0
                    nor4 = 0
                    nor5 = 0
                    nor6 = 0
                    nor7 = 0
                    nor8 = 0
                    for u in range(len(X)):
                        if (Data[u][m][tau][0] >= WallComplexity and Data[u][m][tau][1] >= WallEntropy and Data[u][m][tau][2] >= WallHurst):
                            if (Y[u] == b"Abnormal"):
                                abn1 += 1
                            else:
                                nor1 += 1
                        elif (Data[u][m][tau][0] >= WallComplexity and Data[u][m][tau][1] >= WallEntropy and Data[u][m][tau][2] < WallHurst):
                            if (Y[u] == b"Abnormal"):
                                abn2 += 1
                            else:
                                nor2 += 1
                        elif (Data[u][m][tau][0] >= WallComplexity and Data[u][m][tau][1] < WallEntropy and Data[u][m][tau][2] >= WallHurst):
                            if (Y[u] == b"Abnormal"):
                                abn3 += 1
                            else:
                                nor3 += 1
                        elif (Data[u][m][tau][0] >= WallComplexity and Data[u][m][tau][1] < WallEntropy and Data[u][m][tau][2] < WallHurst):
                            if (Y[u] == b"Abnormal"):
                                abn4 += 1
                            else:
                                nor4 += 1
                        elif (Data[u][m][tau][0] < WallComplexity and Data[u][m][tau][1] >= WallEntropy and Data[u][m][tau][2] >= WallHurst):
                            if (Y[u] == b"Abnormal"):
                                abn5 += 1
                            else:
                                nor5 += 1
                        elif (Data[u][m][tau][0] < WallComplexity and Data[u][m][tau][1] >= WallEntropy and Data[u][m][tau][2] < WallHurst):
                            if (Y[u] == b"Abnormal"):
                                abn6 += 1
                            else:
                                nor6 += 1
                        elif (Data[u][m][tau][0] < WallComplexity and Data[u][m][tau][1] < WallEntropy and Data[u][m][tau][2] >= WallHurst):
                            if (Y[u] == b"Abnormal"):
                                abn7 += 1
                            else:
                                nor7 += 1
                        else:
                            if (Y[u] == b"Abnormal"):
                                abn8 += 1
                            else:
                                nor8 += 1
                
                    for u in range(256):
                        abn = 0
                        nor = 0

                        if u % 2 == 0:
                            abn = abn + abn1
                        else:
                            nor = nor + nor1
                        if (u // 2) % 2 == 0:
                            abn = abn + abn2
                        else:
                            nor = nor + nor2
                        if (u // 4) % 2 == 0:
                            abn = abn + abn3
                        else:
                            nor = nor + nor3
                        if (u // 8) % 2 == 0:
                            abn = abn + abn4
                        else:
                            nor = nor + nor4
                        if (u // 16) % 2 == 0:
                            abn = abn + abn5
                        else:
                            nor = nor + nor5
                        if (u // 32) % 2 == 0:
                            abn = abn + abn6
                        else:
                            nor = nor + nor6
                        if (u // 64) % 2 == 0:
                            abn = abn + abn7
                        else:
                            nor = nor + nor7
                        if (u // 128) % 2 == 0:
                            abn = abn + abn8
                        else:
                            nor = nor + nor8

                        if (abn / 150 * 100 + nor / 55 * 100 > maximum):
                            maximum = abn / 150 * 100 + nor / 55 * 100
                            print(abn / 150 * 100, nor / 55 * 100, abn, nor, u, m, tau, WallEntropy, WallComplexity, WallHurst)
                        if (abn == 150 and nor > maximumA):
                            maximumA = nor
                            print(abn / 150 * 100, nor / 55 * 100, abn, nor, u, m, tau, WallEntropy, WallComplexity, WallHurst)
                        if (nor == 55 and abn > maximumN):
                            maximumN = abn
                            print(abn / 150 * 100, nor / 55 * 100, abn, nor, u, m, tau, WallEntropy, WallComplexity, WallHurst)

1
100.0 0.0 150 0 0 4 1 0.7057468862751003 0.2214487550572745 0.2827887150170419
88.66666666666667 14.545454545454545 133 8 4 4 1 0.7057468862751003 0.2214487550572745 0.2827887150170419
51.33333333333333 70.9090909090909 77 39 8 4 1 0.7057468862751003 0.2214487550572745 0.2827887150170419
40.0 85.45454545454545 60 47 12 4 1 0.7057468862751003 0.2214487550572745 0.2827887150170419
100.0 1.8181818181818181 150 1 64 4 1 0.7057468862751003 0.2214487550572745 0.2827887150170419
40.0 87.27272727272727 60 48 76 4 1 0.7057468862751003 0.2214487550572745 0.2827887150170419
2.666666666666667 100.0 4 55 124 4 1 0.7057468862751003 0.2214487550572745 0.2827887150170419
44.666666666666664 83.63636363636363 67 46 8 4 1 0.7057468862751003 0.2214487550572745 0.30911023384621006
44.0 85.45454545454545 66 47 136 4 1 0.7057468862751003 0.2214487550572745 0.30911023384621006
20.0 100.0 30 55 172 4 1 0.7057468862751003 0.2214487550572745 0.30911023384621006
100.0 3.6363636363636362 150 2 8 4 1 0.7057468862

In [ ]:
print(maximum, maximumN, max)

142.36363636363637 41 9


In [24]:
for m in range(5, 6):
    for tau in range(1, 5):
        print(tau)
        for i in range(len(X)):
            for j in range(len(X)):
                for o in range(len(X)):
                    WallComplexity = Data[i][m][tau][0]
                    WallEntropy = Data[j][m][tau][1]
                    WallHurst = Data[o][m][tau][2]

                    abn1 = 0
                    abn2 = 0
                    abn3 = 0
                    abn4 = 0
                    abn5 = 0
                    abn6 = 0
                    abn7 = 0
                    abn8 = 0
                    nor1 = 0
                    nor2 = 0
                    nor3 = 0
                    nor4 = 0
                    nor5 = 0
                    nor6 = 0
                    nor7 = 0
                    nor8 = 0
                    for u in range(len(X)):
                        if (Data[u][m][tau][0] >= WallComplexity and Data[u][m][tau][1] >= WallEntropy and Data[u][m][tau][2] >= WallHurst):
                            if (Y[u] == b"Abnormal"):
                                abn1 += 1
                            else:
                                nor1 += 1
                        elif (Data[u][m][tau][0] >= WallComplexity and Data[u][m][tau][1] >= WallEntropy and Data[u][m][tau][2] < WallHurst):
                            if (Y[u] == b"Abnormal"):
                                abn2 += 1
                            else:
                                nor2 += 1
                        elif (Data[u][m][tau][0] >= WallComplexity and Data[u][m][tau][1] < WallEntropy and Data[u][m][tau][2] >= WallHurst):
                            if (Y[u] == b"Abnormal"):
                                abn3 += 1
                            else:
                                nor3 += 1
                        elif (Data[u][m][tau][0] >= WallComplexity and Data[u][m][tau][1] < WallEntropy and Data[u][m][tau][2] < WallHurst):
                            if (Y[u] == b"Abnormal"):
                                abn4 += 1
                            else:
                                nor4 += 1
                        elif (Data[u][m][tau][0] < WallComplexity and Data[u][m][tau][1] >= WallEntropy and Data[u][m][tau][2] >= WallHurst):
                            if (Y[u] == b"Abnormal"):
                                abn5 += 1
                            else:
                                nor5 += 1
                        elif (Data[u][m][tau][0] < WallComplexity and Data[u][m][tau][1] >= WallEntropy and Data[u][m][tau][2] < WallHurst):
                            if (Y[u] == b"Abnormal"):
                                abn6 += 1
                            else:
                                nor6 += 1
                        elif (Data[u][m][tau][0] < WallComplexity and Data[u][m][tau][1] < WallEntropy and Data[u][m][tau][2] >= WallHurst):
                            if (Y[u] == b"Abnormal"):
                                abn7 += 1
                            else:
                                nor7 += 1
                        else:
                            if (Y[u] == b"Abnormal"):
                                abn8 += 1
                            else:
                                nor8 += 1
                
                    for u in range(256):
                        abn = 0
                        nor = 0

                        if u % 2 == 0:
                            abn = abn + abn1
                        else:
                            nor = nor + nor1
                        if (u // 2) % 2 == 0:
                            abn = abn + abn2
                        else:
                            nor = nor + nor2
                        if (u // 4) % 2 == 0:
                            abn = abn + abn3
                        else:
                            nor = nor + nor3
                        if (u // 8) % 2 == 0:
                            abn = abn + abn4
                        else:
                            nor = nor + nor4
                        if (u // 16) % 2 == 0:
                            abn = abn + abn5
                        else:
                            nor = nor + nor5
                        if (u // 32) % 2 == 0:
                            abn = abn + abn6
                        else:
                            nor = nor + nor6
                        if (u // 64) % 2 == 0:
                            abn = abn + abn7
                        else:
                            nor = nor + nor7
                        if (u // 128) % 2 == 0:
                            abn = abn + abn8
                        else:
                            nor = nor + nor8

                        if (abn / 150 * 100 + nor / 55 * 100 > maximum):
                            maximum = abn / 150 * 100 + nor / 55 * 100
                            print(abn / 150 * 100, nor / 55 * 100, abn, nor, u, m, tau, WallEntropy, WallComplexity, WallHurst)
                        if (abn == 150 and nor > maximumA):
                            maximumA = nor
                            print(abn / 150 * 100, nor / 55 * 100, abn, nor, u, m, tau, WallEntropy, WallComplexity, WallHurst)
                        if (nor == 55 and abn > maximumN):
                            maximumN = abn
                            print(abn / 150 * 100, nor / 55 * 100, abn, nor, u, m, tau, WallEntropy, WallComplexity, WallHurst)

1
28.000000000000004 100.0 42 55 169 5 1 0.591941578485583 0.27508652126100475 0.30911023384621006
28.666666666666668 100.0 43 55 169 5 1 0.5774118933023508 0.27659386816430775 0.30911023384621006
2
80.66666666666666 61.81818181818181 121 34 37 5 2 0.4944419332509769 0.30840830175298134 0.22321271588740205
81.33333333333333 61.81818181818181 122 34 37 5 2 0.5021624463524575 0.30840830175298134 0.22321271588740205
78.0 65.45454545454545 117 36 97 5 2 0.4859700580306886 0.30840830175298134 0.22512822478945807
72.66666666666667 70.9090909090909 109 39 99 5 2 0.4859700580306886 0.30840830175298134 0.23212509175309073
77.33333333333333 67.27272727272727 116 37 97 5 2 0.4859700580306886 0.30840830175298134 0.22321271588740205
78.0 67.27272727272727 117 37 97 5 2 0.4885142821936163 0.30840830175298134 0.22321271588740205
82.0 63.63636363636363 123 35 97 5 2 0.4885142821936163 0.3099475830338496 0.22321271588740205
82.66666666666667 63.63636363636363 124 35 97 5 2 0.4885142821936163 0.31008412

In [27]:
print(maximum, maximumN, maximumA)

146.3030303030303 43 9


In [28]:
for m in range(6, 7):
    for tau in range(1, 5):
        print(tau)
        for i in range(len(X)):
            for j in range(len(X)):
                for o in range(len(X)):
                    WallComplexity = Data[i][m][tau][0]
                    WallEntropy = Data[j][m][tau][1]
                    WallHurst = Data[o][m][tau][2]

                    abn1 = 0
                    abn2 = 0
                    abn3 = 0
                    abn4 = 0
                    abn5 = 0
                    abn6 = 0
                    abn7 = 0
                    abn8 = 0
                    nor1 = 0
                    nor2 = 0
                    nor3 = 0
                    nor4 = 0
                    nor5 = 0
                    nor6 = 0
                    nor7 = 0
                    nor8 = 0
                    for u in range(len(X)):
                        if (Data[u][m][tau][0] >= WallComplexity and Data[u][m][tau][1] >= WallEntropy and Data[u][m][tau][2] >= WallHurst):
                            if (Y[u] == b"Abnormal"):
                                abn1 += 1
                            else:
                                nor1 += 1
                        elif (Data[u][m][tau][0] >= WallComplexity and Data[u][m][tau][1] >= WallEntropy and Data[u][m][tau][2] < WallHurst):
                            if (Y[u] == b"Abnormal"):
                                abn2 += 1
                            else:
                                nor2 += 1
                        elif (Data[u][m][tau][0] >= WallComplexity and Data[u][m][tau][1] < WallEntropy and Data[u][m][tau][2] >= WallHurst):
                            if (Y[u] == b"Abnormal"):
                                abn3 += 1
                            else:
                                nor3 += 1
                        elif (Data[u][m][tau][0] >= WallComplexity and Data[u][m][tau][1] < WallEntropy and Data[u][m][tau][2] < WallHurst):
                            if (Y[u] == b"Abnormal"):
                                abn4 += 1
                            else:
                                nor4 += 1
                        elif (Data[u][m][tau][0] < WallComplexity and Data[u][m][tau][1] >= WallEntropy and Data[u][m][tau][2] >= WallHurst):
                            if (Y[u] == b"Abnormal"):
                                abn5 += 1
                            else:
                                nor5 += 1
                        elif (Data[u][m][tau][0] < WallComplexity and Data[u][m][tau][1] >= WallEntropy and Data[u][m][tau][2] < WallHurst):
                            if (Y[u] == b"Abnormal"):
                                abn6 += 1
                            else:
                                nor6 += 1
                        elif (Data[u][m][tau][0] < WallComplexity and Data[u][m][tau][1] < WallEntropy and Data[u][m][tau][2] >= WallHurst):
                            if (Y[u] == b"Abnormal"):
                                abn7 += 1
                            else:
                                nor7 += 1
                        else:
                            if (Y[u] == b"Abnormal"):
                                abn8 += 1
                            else:
                                nor8 += 1
                
                    for u in range(256):
                        abn = 0
                        nor = 0

                        if u % 2 == 0:
                            abn = abn + abn1
                        else:
                            nor = nor + nor1
                        if (u // 2) % 2 == 0:
                            abn = abn + abn2
                        else:
                            nor = nor + nor2
                        if (u // 4) % 2 == 0:
                            abn = abn + abn3
                        else:
                            nor = nor + nor3
                        if (u // 8) % 2 == 0:
                            abn = abn + abn4
                        else:
                            nor = nor + nor4
                        if (u // 16) % 2 == 0:
                            abn = abn + abn5
                        else:
                            nor = nor + nor5
                        if (u // 32) % 2 == 0:
                            abn = abn + abn6
                        else:
                            nor = nor + nor6
                        if (u // 64) % 2 == 0:
                            abn = abn + abn7
                        else:
                            nor = nor + nor7
                        if (u // 128) % 2 == 0:
                            abn = abn + abn8
                        else:
                            nor = nor + nor8

                        if (abn / 150 * 100 + nor / 55 * 100 > maximum):
                            maximum = abn / 150 * 100 + nor / 55 * 100
                            print(abn / 150 * 100, nor / 55 * 100, abn, nor, u, m, tau, WallEntropy, WallComplexity, WallHurst)
                        if (abn == 150 and nor > maximumA):
                            maximumA = nor
                            print(abn / 150 * 100, nor / 55 * 100, abn, nor, u, m, tau, WallEntropy, WallComplexity, WallHurst)
                        if (nor == 55 and abn > maximumN):
                            maximumN = abn
                            print(abn / 150 * 100, nor / 55 * 100, abn, nor, u, m, tau, WallEntropy, WallComplexity, WallHurst)

1
2
3
4
